In [1]:
import opf
from pypglib import pglib_opf_case5_pjm
from matpowercaseframes import CaseFrames
import pandas as pd
import pyomo.environ as pyo  # noqa

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 300)
pd.set_option('display.precision', 2)
pd.set_option('display.float_format', '{:.2f}'.format)

In [3]:
model = opf.build_model('acopf')
network = opf.parse_file(pglib_opf_case5_pjm)
model.instantiate(network)
result = model.solve(
    solver_option={'print_level' : 5},
    tee=True
)

build model... end
instantiate model... end
Ipopt 3.14.16: print_level=5


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.16, running with linear solver MUMPS 5.6.2.

Number of nonzeros in equality constraint Jacobian...:      155
Number of nonzeros in inequality constraint Jacobian.:       36
Number of nonzeros in Lagrangian Hessian.............:       63

Total number of variables............................:       44
                     variables with only lower bounds:        0
                variables with lower and upper bounds:       15
                     variables with only upper bounds:        0
Total number of eq

In [4]:
print(
    f"Status: {result['termination_status']}\n"
    f"Objective function: {result['obj_cost']}\n"
    f"Output power: {result['sol']['primal']['pg']}\n"
    f"Computation time: {result['time']}"
)

Status: optimal
Objective function: 17551.89083859283
Output power: {'1': 0.4000000096583169, '2': 1.700000016481886, '3': 3.2449849431098188, '4': -6.499551075688347e-09, '5': 4.706935997017107}
Computation time: 0.03522610664367676


In [5]:
cf = CaseFrames(pglib_opf_case5_pjm)
cf.infer_numpy()
print(cf.branch)
print(cf.bus)

   F_BUS  T_BUS  BR_R  BR_X  BR_B  RATE_A  RATE_B  RATE_C  TAP  SHIFT  BR_STATUS  ANGMIN  ANGMAX
1      1      2  0.00  0.03  0.01     400     400     400    0      0          1     -30      30
2      1      4  0.00  0.03  0.01     426     426     426    0      0          1     -30      30
3      1      5  0.00  0.01  0.03     426     426     426    0      0          1     -30      30
4      2      3  0.00  0.01  0.02     426     426     426    0      0          1     -30      30
5      3      4  0.00  0.03  0.01     426     426     426    0      0          1     -30      30
6      4      5  0.00  0.03  0.01     240     240     240    0      0          1     -30      30
   BUS_I  BUS_TYPE   PD     QD  GS  BS  BUS_AREA  VM  VA  BASE_KV  ZONE  VMAX  VMIN
1      1         2    0   0.00   0   0         1   1   0      230     1  1.10  0.90
2      2         1  300  98.61   0   0         1   1   0      230     1  1.10  0.90
3      3         2  300  98.61   0   0         1   1   0      230    

In [6]:
def pyo2df(data_names, data_set):
    """
    Create a DataFrame for specified data from a Pyomo model.

    Parameters:
    - data_names: list of str, names of the data attributes (e.g., ["pg", "qg"])
    - data_set: set, the set of indices to iterate over

    Returns:
    - DataFrame: pandas DataFrame containing the data.
    """
    data = []
    for idx in data_set:
        row_data = {}
        for name in data_names:
            value = pyo.value(getattr(model.instance, name)[idx])
            row_data[name] = value
        data.append(row_data)

    return pd.DataFrame(data, index=list(data_set))

In [7]:
data = []
for idx in model.instance.E:
    pf_from = pyo.value(model.instance.pf_from[idx])
    pf_to = pyo.value(model.instance.pf_to[idx])
    losses = abs(pf_from - pf_to)

    # print(f"{model.instance.pf_from[idx]}: {pf_from}")
    # print(f"{model.instance.pf_to[idx]}: {pf_to}")
    # print(f"losses[{str(idx)}]: {losses}")

    data.append({
        "pf_from": pf_from,
        "pf_to": pf_to,
        "losses": losses
    })

df_branch = pd.DataFrame(data, index=list(model.instance.E))
print(df_branch)
print(df_branch[['losses']].agg(['sum']))

   pf_from  pf_to  losses
1     2.52  -2.51    5.03
2     1.88  -1.87    3.75
3    -2.30   2.31    4.61
4    -0.49   0.49    0.99
5    -0.25   0.25    0.50
6    -2.39   2.40    4.78
     losses
sum   19.66


In [8]:
data = []
for idx in model.instance.G:
    pg = pyo.value(model.instance.pg[idx])
    qg = pyo.value(model.instance.qg[idx])

    data.append({
        "pg": pg,
        "qg": qg,
    })

df_gen = pd.DataFrame(data, index=list(model.instance.G))
print(df_gen)
print(df_gen.agg(['sum']))

     pg    qg
1  0.40  0.30
2  1.70  1.28
3  3.24  3.90
4 -0.00 -0.11
5  4.71 -1.65
       pg   qg
sum 10.05 3.72


In [9]:
data = []
for idx in model.instance.L:
    pd_ = pyo.value(model.instance.pd[idx])
    data.append({
        "pd": pd_,
    })

df_loads = pd.DataFrame(data, index=list(model.instance.L))
print(df_loads)
print(df_loads.agg(['sum']))

    pd
1 3.00
2 3.00
3 4.00
       pd
sum 10.00


In [10]:
# Collect total load per bus data
data = []
for b in model.instance.B:
    total_load = sum(pyo.value(model.instance.pd[l]) for l in model.instance.load_per_bus[b])
    data.append({
        "pd_per_bus": total_load,
    })

# Create DataFrame for total loads per bus with index
df_bus_loads = pd.DataFrame(data, index=list(model.instance.B))
print(df_bus_loads)
print(df_bus_loads.agg(['sum']))

   pd_per_bus
1        0.00
2        3.00
3        3.00
4        4.00
5        0.00
     pd_per_bus
sum       10.00
